In [1]:
import pandas as pd
fn = "data/cleaned_table_2.parquet"
df = pd.read_parquet(fn)

In [2]:
print(df.shape)
print(df.columns)

(3018, 48)
Index(['X', 'Y', 'Unique Squirrel ID', 'Hectare', 'Shift', 'Date', 'Age',
       'Primary Fur Color', 'Running', 'Chasing', 'Climbing', 'Eating',
       'Foraging', 'Kuks', 'Quaas', 'Moans', 'Tail flags', 'Tail twitches',
       'Approaches', 'Indifferent', 'Runs from',
       'Sighter Observed Weather Data', 'Litter', 'Hectare Conditions',
       'Number of sighters', 'Number of Squirrels', 'Total Time of Sighting',
       'is_weekend', 'is_above_ground', 'above_ground_numeric',
       'location_missing', 'highlight_gray', 'highlight_white',
       'highlight_cinnamon', 'highlight_black', 'highlight_missing',
       'animals_human_present', 'animals_dog_present',
       'animals_pigeon_present', 'animals_bird_present',
       'animals_sparrow_present', 'animals_duck_present',
       'animals_data_missing', 'temperature', 'sky_condition',
       'squirrel_density_proxy', 'session_id', 'y'],
      dtype='object')


In [3]:
# check if there are any missing values in the dataframe, type of every column, and some basic statistics of the numerical columns
print(df.info())
print(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3018 entries, 0 to 3017
Data columns (total 48 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   X                              3018 non-null   float64       
 1   Y                              3018 non-null   float64       
 2   Unique Squirrel ID             3018 non-null   object        
 3   Hectare                        3018 non-null   object        
 4   Shift                          3018 non-null   object        
 5   Date                           3018 non-null   datetime64[ns]
 6   Age                            3018 non-null   object        
 7   Primary Fur Color              3018 non-null   object        
 8   Running                        3018 non-null   bool          
 9   Chasing                        3018 non-null   bool          
 10  Climbing                       3018 non-null   bool          
 11  Eating           

In [4]:
# extract numerical columns
numerical_cols = df.select_dtypes(include=["float64", "int64"]).columns
print("Numerical columns:", numerical_cols)

# extract categorical columns
categorical_cols = df.select_dtypes(include=["object", "category"]).columns
print("Categorical columns:", categorical_cols)

# extract boolean columns
boolean_cols = df.select_dtypes(include=["bool"]).columns
print("Boolean columns:", boolean_cols)

# check if all the columns are either numerical, categorical, or boolean
all_cols = set(df.columns)
identified_cols = set(numerical_cols) | set(categorical_cols) | set(boolean_cols)
unidentified_cols = all_cols - identified_cols
print("Unidentified columns:", unidentified_cols)

Numerical columns: Index(['X', 'Y', 'Number of sighters', 'Number of Squirrels',
       'Total Time of Sighting', 'above_ground_numeric', 'temperature',
       'squirrel_density_proxy', 'y'],
      dtype='object')
Categorical columns: Index(['Unique Squirrel ID', 'Hectare', 'Shift', 'Age', 'Primary Fur Color',
       'Sighter Observed Weather Data', 'Litter', 'Hectare Conditions',
       'sky_condition', 'session_id'],
      dtype='object')
Boolean columns: Index(['Running', 'Chasing', 'Climbing', 'Eating', 'Foraging', 'Kuks', 'Quaas',
       'Moans', 'Tail flags', 'Tail twitches', 'Approaches', 'Indifferent',
       'Runs from', 'is_weekend', 'is_above_ground', 'location_missing',
       'highlight_gray', 'highlight_white', 'highlight_cinnamon',
       'highlight_black', 'highlight_missing', 'animals_human_present',
       'animals_dog_present', 'animals_pigeon_present', 'animals_bird_present',
       'animals_sparrow_present', 'animals_duck_present',
       'animals_data_missing'],
 

In [5]:
# for numerical columns, list the columns that have missing values
for col in numerical_cols:
    missing_percentage = df[col].isna().mean() * 100
    if missing_percentage > 0:
        print(f"{col}: {missing_percentage:.2f}% missing values")

# fix above_ground_numeric's missing
# fill 0 to missing values in above_ground_numeric if location_missing is True
df.loc[df["location_missing"] == True, "above_ground_numeric"] = df.loc[df["location_missing"] == True, "above_ground_numeric"].fillna(0)
# extract rows where is_above_ground is True and fill missing values in above_ground_numeric with median
median_value = df.loc[df["is_above_ground"] == True, "above_ground_numeric"].median()
df.loc[df["is_above_ground"] == True, "above_ground_numeric"] = df.loc[df["is_above_ground"] == True, "above_ground_numeric"].fillna(median_value)
# check if there are still missing values in above_ground_numeric
missing_percentage = df["above_ground_numeric"].isna().mean() * 100
print(f"above_ground_numeric: {missing_percentage:.2f}% missing values")


above_ground_numeric: 3.78% missing values
y: 73.03% missing values
above_ground_numeric: 0.00% missing values


In [6]:
# for categorical columns, list the columns that have missing values and the percentage of missing values
for col in categorical_cols:
    missing_percentage = df[col].isna().mean() * 100
    if missing_percentage > 0:
        print(f"{col}: {missing_percentage:.2f}% missing values")

# fix Litter missing values by filling "Unknown"
df["Litter"] = df["Litter"].fillna("Unknown")
# check if there are still missing values in Litter
missing_percentage = df["Litter"].isna().mean() * 100
print(f"Litter: {missing_percentage:.2f}% missing values")

# for sky_condition, fill missing values with "Unknown"
df["sky_condition"] = df["sky_condition"].fillna("Unknown")
# check if there are still missing values in sky_condition
missing_percentage = df["sky_condition"].isna().mean() * 100
print(f"sky_condition: {missing_percentage:.2f}% missing values")

Sighter Observed Weather Data: 3.05% missing values
Litter: 12.66% missing values
sky_condition: 17.23% missing values
Litter: 0.00% missing values
sky_condition: 0.00% missing values


In [7]:
# for boolean columns, list the columns that have missing values and the percentage of missing values
for col in boolean_cols:
    missing_percentage = df[col].isna().mean() * 100
    if missing_percentage > 0:
        print(f"{col}: {missing_percentage:.2f}% missing values")

In [8]:
# all missing values have been handled
# now, handle hectare conditions. the variable description already specifies the possible values, so we can encode them as categorical variables
df['Hectare Conditions'].unique()

from sklearn.preprocessing import MultiLabelBinarizer
# split the string by comma and strip the whitespace
df['Hectare Conditions'] = df['Hectare Conditions'].apply(lambda x: [item.strip() for item in x.split(',')])
# use MultiLabelBinarizer to encode the Hectare Conditions column
mlb = MultiLabelBinarizer()
hectare_conditions_encoded = mlb.fit_transform(df['Hectare Conditions'])
# create a dataframe with the encoded hectare conditions and concatenate it with the original dataframe
# named as hectare_condition_<condition>
hectare_conditions_df = pd.DataFrame(hectare_conditions_encoded, columns=mlb.classes_)
hectare_conditions_df.columns = [f"hectare_condition_{col}" for col in hectare_conditions_df.columns]
df = pd.concat([df, hectare_conditions_df], axis=1)
# drop the original Hectare Conditions column
df = df.drop(columns=['Hectare Conditions'])

In [9]:
# these can be one-hot encoded beforehand since they are defined in the description and have a limited number of possible values
ohe_columns = ['Shift', 'Age', 'Primary Fur Color', 'Litter']
# check the unique values of these columns to make sure they are suitable for one-hot encoding
for col in ohe_columns:
    print(f"{col}: {df[col].unique()}")
# one-hot encode the columns and concatenate with the original dataframe
df = pd.get_dummies(df, columns=ohe_columns, prefix=ohe_columns)

Shift: ['PM' 'AM']
Age: ['Adult' 'Juvenile']
Primary Fur Color: ['Gray' 'Cinnamon' 'Black']
Litter: ['None' 'Some' 'Unknown' 'Abundant']


In [10]:
# check the remaining categorical columns and decide how to encode them
remaining_categorical_cols = df.select_dtypes(include=["object", "category"]).columns
print("Remaining categorical columns:", remaining_categorical_cols)

Remaining categorical columns: Index(['Unique Squirrel ID', 'Hectare', 'Sighter Observed Weather Data',
       'sky_condition', 'session_id'],
      dtype='object')


In [12]:
from modelling.weather_category import weather_category
weather_raw = df["Sighter Observed Weather Data"]
df['weather_condition'] = weather_category(weather_raw)


In [13]:
df[['weather_condition', 'sky_condition']]
# check the unknown percentage in weather_condition and sky_condition
print(df['weather_condition'].value_counts(normalize=True) * 100) # less unknown values than sky_condition, so we can keep it as a feature
print(df['sky_condition'].value_counts(normalize=True) * 100)

weather_condition
cloudy     38.966203
sunny      29.158383
rainy      17.925779
unknown    13.949636
Name: proportion, dtype: float64
sky_condition
cloudy        31.875414
sunny         24.320742
Unknown       17.229954
overcast      11.530815
clear          4.970179
mist           4.009278
rainy          2.584493
fog            1.789264
drizzle        0.927767
sprinkling     0.596421
wet            0.165673
Name: proportion, dtype: float64


In [14]:
# check missing values again to make sure all missing values have been handled
print(df.isna().sum().sort_values(ascending=False))

y                                2204
Sighter Observed Weather Data      92
Unique Squirrel ID                  0
Hectare                             0
Date                                0
Running                             0
Chasing                             0
Climbing                            0
Eating                              0
X                                   0
Foraging                            0
Kuks                                0
Moans                               0
Quaas                               0
Tail twitches                       0
Approaches                          0
Indifferent                         0
Tail flags                          0
Runs from                           0
Number of sighters                  0
Number of Squirrels                 0
Total Time of Sighting              0
is_weekend                          0
is_above_ground                     0
above_ground_numeric                0
Y                                   0
location_mis

In [15]:
# extract numerical columns
numerical_cols = df.select_dtypes(include=["float64", "int64"]).columns
print("Numerical columns:", numerical_cols)

# extract categorical columns
categorical_cols = df.select_dtypes(include=["object", "category"]).columns
print("Categorical columns:", categorical_cols)

# extract boolean columns
boolean_cols = df.select_dtypes(include=["bool"]).columns
print("Boolean columns:", boolean_cols)

# check if all the columns are either numerical, categorical, or boolean
all_cols = set(df.columns)
identified_cols = set(numerical_cols) | set(categorical_cols) | set(boolean_cols)
unidentified_cols = all_cols - identified_cols
print("Unidentified columns:", unidentified_cols)

Numerical columns: Index(['X', 'Y', 'Number of sighters', 'Number of Squirrels',
       'Total Time of Sighting', 'above_ground_numeric', 'temperature',
       'squirrel_density_proxy', 'y', 'hectare_condition_Busy',
       'hectare_condition_Calm', 'hectare_condition_Moderate'],
      dtype='object')
Categorical columns: Index(['Unique Squirrel ID', 'Hectare', 'Sighter Observed Weather Data',
       'sky_condition', 'session_id', 'weather_condition'],
      dtype='object')
Boolean columns: Index(['Running', 'Chasing', 'Climbing', 'Eating', 'Foraging', 'Kuks', 'Quaas',
       'Moans', 'Tail flags', 'Tail twitches', 'Approaches', 'Indifferent',
       'Runs from', 'is_weekend', 'is_above_ground', 'location_missing',
       'highlight_gray', 'highlight_white', 'highlight_cinnamon',
       'highlight_black', 'highlight_missing', 'animals_human_present',
       'animals_dog_present', 'animals_pigeon_present', 'animals_bird_present',
       'animals_sparrow_present', 'animals_duck_present',

In [17]:
df['Litter_Abundant']

0       False
1       False
2       False
3       False
4       False
        ...  
3013    False
3014    False
3015    False
3016    False
3017    False
Name: Litter_Abundant, Length: 3018, dtype: bool